In [ ]:
import numpy as np
from sklearn.utils import class_weight
import h5py

import tensorflow as tf
tf.keras.utils.set_random_seed(424)

import warnings
warnings.filterwarnings("ignore")

In [2]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if len(gpus):
    # 设置 GPU 显存占用为按需分配，增长式
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
        	# 异常处理
        	print(e)

In [ ]:
# all subjects PPG data and ann data
with h5py.File(f"sig_mat_2064_1.h5", "r") as rf :
    sig_mat = rf['sig_mat'][:][:,::,1:]
with h5py.File(f"ann_seg_2064_1.h5", "r") as rf :
    ann_seg = rf['ann_seg'][:]

In [4]:
# V3, V2, A3, A2, -V3, -V2, -A3, -A2, VA4, VA5, -VA4, -VA5, L3
# 0,  1,  2,  3,  4,   5,   6,   7,   8,   9,   10,   11,   12

lab_seg = np.zeros((len(ann_seg[:,0]),13))
for line in np.arange(len(ann_seg[:,0])) :
    # ====== V temp =======
    if ann_seg[line,0] <= 3.5 :
        lab_seg[line,0] = 0
        lab_seg[line,1] = 0
    elif ann_seg[line,0] <= 6.5 :
        lab_seg[line,0] = 1
        if ann_seg[line,0] <= 5 :
            lab_seg[line,1] = 0
        else:
            lab_seg[line,1] = 1
    else :
        lab_seg[line,0] = 2
        lab_seg[line,1] = 1
    # ====== A temp =======    
    if ann_seg[line,1] <= 3.5 :
        lab_seg[line,2] = 0
        lab_seg[line,3] = 0
    elif ann_seg[line,1] <= 6.5 :
        lab_seg[line,2] = 1
        if ann_seg[line,1] <= 5 :
            lab_seg[line,3] = 0
        else:
            lab_seg[line,3] = 1
    else :
        lab_seg[line,2] = 2
        lab_seg[line,3] = 1

    # ====== V avg =======
    if ann_seg[line,2] <= 3.5 :
        lab_seg[line,4] = 0
        lab_seg[line,5] = 0
    elif ann_seg[line,2] <= 6.5 :
        lab_seg[line,4] = 1
        if ann_seg[line,2] <= 5 :
            lab_seg[line,5] = 0
        else:
            lab_seg[line,5] = 1
    else :
        lab_seg[line,4] = 2
        lab_seg[line,5] = 1
    # ====== A avg =======    
    if ann_seg[line,3] <= 3.5 :
        lab_seg[line,6] = 0
        lab_seg[line,7] = 0
    elif ann_seg[line,3] <= 6.5 :
        lab_seg[line,6] = 1
        if ann_seg[line,3] <= 5 :
            lab_seg[line,7] = 0
        else:
            lab_seg[line,7] = 1
    else :
        lab_seg[line,6] = 2
        lab_seg[line,7] = 1

    # ======= VA-4 temp ======
    if lab_seg[line,1] == 0 and lab_seg[line,3] == 0 :
        lab_seg[line,8] = 1
    elif lab_seg[line,1] == 0 and lab_seg[line,3] == 1 :
        lab_seg[line,8] = 0
    elif lab_seg[line,1] == 1 and lab_seg[line,3] == 0 :
        lab_seg[line,8] = 2
    elif lab_seg[line,1] == 1 and lab_seg[line,3] == 1 :
        lab_seg[line,8] = 3
    # ======== VA-4 avg ======
    if lab_seg[line,5] == 0 and lab_seg[line,7] == 0 :
        lab_seg[line,10] = 1
    elif lab_seg[line,5] == 0 and lab_seg[line,7] == 1 :
        lab_seg[line,10] = 0
    elif lab_seg[line,5] == 1 and lab_seg[line,7] == 0 :
        lab_seg[line,10] = 2
    elif lab_seg[line,5] == 1 and lab_seg[line,7] == 1 :
        lab_seg[line,10] = 3 
    
    # ======== VA-5 temp ======
    if lab_seg[line,0] == 1 and lab_seg[line,2] == 1 :
        lab_seg[line,9] = 2
    elif lab_seg[line,1] == 0 and lab_seg[line,3] == 0 :
        lab_seg[line,9] = 1
    elif lab_seg[line,1] == 0 and lab_seg[line,3] == 1 :
        lab_seg[line,9] = 0
    elif lab_seg[line,1] == 1 and lab_seg[line,3] == 0 :
        lab_seg[line,9] = 3
    elif lab_seg[line,1] == 1 and lab_seg[line,3] == 1 :
        lab_seg[line,9] = 4
    # ======== VA-4 avg ======
    if lab_seg[line,4] == 1 and lab_seg[line,6] == 1 :
        lab_seg[line,11] = 2
    elif lab_seg[line,5] == 0 and lab_seg[line,7] == 0 :
        lab_seg[line,11] = 1
    elif lab_seg[line,5] == 0 and lab_seg[line,7] == 1 :
        lab_seg[line,11] = 0
    elif lab_seg[line,5] == 1 and lab_seg[line,7] == 0 :
        lab_seg[line,11] = 3
    elif lab_seg[line,5] == 1 and lab_seg[line,7] == 1 :
        lab_seg[line,11] = 4 

    # ======== L3 =========
    if ann_seg[line,5] ==0 or ann_seg[line,5] == 1 :
        lab_seg[line,12] = 2
    elif ann_seg[line,5] ==6 or ann_seg[line,5] == 7 :
        lab_seg[line,12] = 0
    else:
        lab_seg[line,12] = 1

In [5]:
class FFTLayer1d(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(FFTLayer1d, self).__init__(**kwargs)

    def call(self, inputs):
        # 对每个 channel 进行 FFT 变换
        fft_result1 = tf.signal.fft(tf.cast(inputs[:,:,0], tf.complex64))
        # 将实部和虚部沿着最后一个维度拼接
        amp_norm = np.ones((1,1280))/640
        amp_norm[:,0] = 0
        real_part1 = tf.math.real(fft_result1)*amp_norm
        imag_part1 = tf.math.imag(fft_result1)*amp_norm
        
        return tf.concat([real_part1[:,:,tf.newaxis], imag_part1[:,:,tf.newaxis]], axis=-1)

    def compute_output_shape(self, input_shape):
        # 输出维度为 (batch, length, 2 * channel)
        return (input_shape[0], input_shape[1], 2 * input_shape[2])

In [6]:
class FFTLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(FFTLayer, self).__init__(**kwargs)

    def call(self, inputs):
        # 对每个 channel 进行 FFT 变换
        fft_result1 = tf.signal.fft(tf.cast(inputs[:,:,0], tf.complex64))
        # 将实部和虚部沿着最后一个维度拼接
        amp_norm = np.ones((1,1280))/640
        amp_norm[:,0] = 0
        real_part1 = tf.math.real(fft_result1)*amp_norm
        imag_part1 = tf.math.imag(fft_result1)*amp_norm

        # 对每个 channel 进行 FFT 变换
        fft_result2 = tf.signal.fft(tf.cast(inputs[:,:,1], tf.complex64))
        # 将实部和虚部沿着最后一个维度拼接
        amp_norm = np.ones((1,1280))/640
        amp_norm[:,0] = 0
        real_part2 = tf.math.real(fft_result2)*amp_norm
        imag_part2 = tf.math.imag(fft_result2)*amp_norm

        # 对每个 channel 进行 FFT 变换
        fft_result3 = tf.signal.fft(tf.cast(inputs[:,:,2], tf.complex64))
        # 将实部和虚部沿着最后一个维度拼接
        amp_norm = np.ones((1,1280))/640
        amp_norm[:,0] = 0
        real_part3 = tf.math.real(fft_result3)*amp_norm
        imag_part3 = tf.math.imag(fft_result3)*amp_norm
        
        return tf.concat([real_part1[:,:,tf.newaxis], imag_part1[:,:,tf.newaxis],
                          real_part2[:,:,tf.newaxis], imag_part2[:,:,tf.newaxis],
                          real_part3[:,:,tf.newaxis], imag_part3[:,:,tf.newaxis]], axis=-1)

    def compute_output_shape(self, input_shape):
        # 输出维度为 (batch, length, 2 * channel)
        return (input_shape[0], input_shape[1], 2 * input_shape[2])


In [7]:
class FFTLayer1D(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(FFTLayer1D, self).__init__(**kwargs)

    def call(self, inputs):
        # 对每个 channel 进行 FFT 变换
        fft_result1 = tf.signal.fft(tf.cast(inputs[:,:,0], tf.complex64))
        # 将实部和虚部沿着最后一个维度拼接
        real_part1 = tf.math.real(fft_result1)
        imag_part1 = tf.math.imag(fft_result1)
        
        return tf.concat([real_part1[:,:,tf.newaxis], imag_part1[:,:,tf.newaxis]], axis=-1)

    def compute_output_shape(self, input_shape):
        # 输出维度为 (batch, length, 2 * channel)
        return (input_shape[0], input_shape[1], 2 * input_shape[2])

In [8]:
class IFFTLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(IFFTLayer, self).__init__(**kwargs)

    def call(self, inputs):
        # 将输入分为实部和虚部
        real_part = inputs[..., 0]
        imag_part = inputs[..., 1]
        # 组合成复数
        complex_input = tf.complex(real_part, imag_part)
        # 对每个 channel 进行 IFFT 变换
        ifft_result = tf.signal.ifft(complex_input)
        return tf.math.real(ifft_result)[:,:,tf.newaxis]

    def compute_output_shape(self, input_shape):
        # 输出维度为 (batch, length, channel)
        return (input_shape[0], input_shape[1], input_shape[2] // 2)


In [9]:
class CrossTFAttention(tf.keras.layers.Layer):
    def __init__(self, attnum, num_head, key_dim, output_shape, **kwargs):
        super(CrossTFAttention, self).__init__(name=f'crftatt{attnum}_block', **kwargs)
        self.frqcov = tf.keras.layers.Conv1D(
            filters=output_shape,
            kernel_size=3,
            padding='same',
            name=f'crftatt{attnum}_frqcov')

        self.ifft = IFFTLayer()
        self.outshape = output_shape
        # self.mhatt = tf.keras.layers.MultiHeadAttention(num_heads=num_head,key_dim=key_dim,output_shape=output_shape,dropout=0.1,name=f'crftatt{attnum}_mhatt')
        self.add = tf.keras.layers.Add(name=f'crftatt{attnum}_mhatt')
        # self.attadd = tf.keras.layers.Add(name=f'crftatt{attnum}_add')

    def call(self, frq, sig) :
        frq_components = tf.split(frq, num_or_size_splits=int(self.outshape/2), axis=-1)
        frq_outputs = []
        for i in range(int(self.outshape/2)) :
            frq_sig = self.ifft(frq_components[i])
            frq_outputs.append(frq_sig)
        ftime_sig = tf.concat(frq_outputs, axis=2)
        frq_cp_i = self.frqcov(ftime_sig)
        # timfreq_att = self.mhatt(query=frq_cp_i,value=sig,key=sig)
        # resadd_att = self.attadd([sig,timfreq_att,frq_cp_i])
        resadd_att = self.add([sig,frq_cp_i])
        
        return resadd_att 

In [10]:
class MultiConv1DLayer(tf.keras.layers.Layer):
    def __init__(self, cov_size, splits_size, trainable=True, filters=1, kernel_size=1, activation='gelu', **kwargs):
        """
        自定义多 Conv1D 层
        
        参数:
        cov_size -- 表示要使用的 Conv1D 层的数量
        filters -- 每个 Conv1D 层的过滤器数量，默认为32
        kernel_size -- 每个 Conv1D 层的核大小，默认为3
        activation -- 每个 Conv1D 层的激活函数，默认为'relu'
        """
        super(MultiConv1DLayer, self).__init__(**kwargs)
        self.cov_size = cov_size
        self.splits_size = splits_size
        self.filters = filters
        self.kernel_size = kernel_size
        self.activation = activation
        self.conv_layers = []
        self.trainable = trainable

    def build(self, input_shape):
        # 创建 cov_size 个 Conv1D 层
        for _ in range(self.cov_size):
            self.conv_layers.append(tf.keras.layers.Conv1D(
                filters=self.filters,
                kernel_size=self.kernel_size,
                padding="same",
                kernel_initializer=tf.keras.initializers.HeNormal(),
                trainable = self.trainable,
                activation=self.activation
            ))
        super(MultiConv1DLayer, self).build(input_shape)

    def call(self, inputs):               
        # 每个 Conv1D 层处理输入的一个分量
        outputs = []
        if self.splits_size == 3 :
            # 分割输入张量，使得每个 Conv1D 层可以处理输入的不同分量
            input_components = tf.split(inputs, num_or_size_splits=3, axis=2) 
            for i in range(self.cov_size - 2):
                output_i = self.conv_layers[i](input_components[int(i/2)])# + input_components[int(i/2)]
                outputs.append(output_i)
            outputs.append(self.conv_layers[-2](inputs))
            outputs.append(self.conv_layers[-1](inputs))
        elif self.splits_size == 6 :
            # 分割输入张量，使得每个 Conv1D 层可以处理输入的不同分量
            input_components = tf.split(inputs, num_or_size_splits=6, axis=2) 
            for i in range(self.cov_size - 2):
                output_i = self.conv_layers[i](input_components[i])# + input_components[i]
                outputs.append(output_i)
            outputs.append(self.conv_layers[-2](inputs[:,:,0:6:2]))
            outputs.append(self.conv_layers[-1](inputs[:,:,1:6:2]))
        else :
            # 分割输入张量，使得每个 Conv1D 层可以处理输入的不同分量
            input_components = tf.split(inputs, num_or_size_splits=self.splits_size, axis=2) 
            for i in range(self.cov_size - 1):
                output_i = self.conv_layers[i](input_components[int(i/2)])# + input_components[int(i/2)]
                outputs.append(output_i)
            outputs.append(self.conv_layers[-1](inputs))
        
        # 拼接所有 Conv1D 层的输出
        return tf.concat(outputs, axis=2)
    
    def get_config(self):
        config = super(MultiConv1DLayer, self).get_config()
        config.update({
            'cov_size': self.cov_size,
            'splits_size': self.splits_size,
            'filters': self.filters,
            'kernel_size': self.kernel_size,
            'activation': self.activation
        })
        return config

In [11]:
class ResBlock1D(tf.keras.layers.Layer):
    def __init__(self, covfilter, resnum, splits_size, covks=3, covstrides=1, rb_trainable=True, **kwargs):
        super(ResBlock1D, self).__init__(name=f'rb{resnum}_block', **kwargs)

        self.rescov = MultiConv1DLayer(
            cov_size=covfilter,
            splits_size = splits_size,
            trainable=rb_trainable,
            name=f'rb{resnum}_rescov'
        )

        self.res1_bn1 = tf.keras.layers.LayerNormalization(name=f'rb{resnum}_res1_bn1')
        self.res1_act1 = tf.keras.layers.Activation('gelu',name=f'rb{resnum}_res1_act1')
        self.res1_cov1 = tf.keras.layers.Conv1D(
            filters=covfilter,
            kernel_size=covks,
            padding='same',
            dilation_rate=1,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            trainable=rb_trainable,
            name=f'rb{resnum}_res1_cov1'
        )
        self.res1_bn2 = tf.keras.layers.LayerNormalization(name=f'rb{resnum}_res1_bn2')
        self.res1_act2 = tf.keras.layers.Activation('gelu',name=f'rb{resnum}_res1_act2')        
        self.res1_cov2 = tf.keras.layers.Conv1D(
            filters=covfilter,
            kernel_size=covks,
            padding='same',
            dilation_rate=2,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            trainable=rb_trainable,
            name=f'rb{resnum}_res1_cov2'
        )
        self.add1 = tf.keras.layers.Add(name=f'rb{resnum}_res1_add')

        self.res2_bn1 = tf.keras.layers.LayerNormalization(name=f'rb{resnum}_res2_bn1')
        self.res2_act1 = tf.keras.layers.Activation('gelu',name=f'rb{resnum}_res2_act1')
        self.res2_cov1 = tf.keras.layers.Conv1D(
            filters=covfilter,
            kernel_size=covks,
            dilation_rate=3,
            padding='same',
            kernel_initializer=tf.keras.initializers.HeNormal(),
            trainable=rb_trainable,
            name=f'rb{resnum}_res2_cov1'
        )
        self.res2_bn2 = tf.keras.layers.LayerNormalization(name=f'rb{resnum}_res2_bn2')
        self.res2_act2 = tf.keras.layers.Activation('gelu',name=f'rb{resnum}_res2_act2')        
        self.res2_cov2 = tf.keras.layers.Conv1D(
            filters=covfilter,
            kernel_size=covks,
            dilation_rate=4,
            padding='same',
            kernel_initializer=tf.keras.initializers.HeNormal(),
            trainable=rb_trainable,
            name=f'rb{resnum}_res2_cov2'
        )
        self.add2 = tf.keras.layers.Add(name=f'rb{resnum}_res2_add')


    def call(self, x):
        rescov = self.rescov(x)
        
        res1_bn1 = self.res1_bn1(rescov)
        res1_act1 = self.res1_act1(res1_bn1)
        res1_cov1 = self.res1_cov1(res1_act1)
        res1_bn2 = self.res1_bn2(res1_cov1)
        res1_act2 = self.res1_act2(res1_bn2)
        res1_cov2 = self.res1_cov2(res1_act2)        
        res1_add = self.add1([rescov, res1_cov2])

        res2_bn1 = self.res2_bn1(res1_add)
        res2_act1 = self.res2_act1(res2_bn1)
        res2_cov1 = self.res2_cov1(res2_act1)
        res2_bn2 = self.res2_bn2(res2_cov1)
        res2_act2 = self.res2_act2(res2_bn2)
        res2_cov2 = self.res2_cov2(res2_act2)        
        res2_add = self.add2([res1_add, res2_cov2])

        return res2_add

In [12]:
def BMAutoEncoder(fs: int=64) -> tf.keras.Model:
    # =========================================== signal layers resUnet ===============================================
    # =========================================== FFT layers resUnet ==================================================
    ipl = tf.keras.Input((20*fs,3))
    FFT_layer = FFTLayer()(ipl)
    # =========================================== signal Encoder ======================================================
    res1 = ResBlock1D(8,1,3)(ipl)
    res1_mp = tf.keras.layers.MaxPool1D(pool_size=4,name="res1_mp")(res1)
    res2 = ResBlock1D(16,2,8,covks=3)(res1_mp)
    res2_mp = tf.keras.layers.MaxPool1D(pool_size=4,name="res2_mp")(res2)
    res3 = ResBlock1D(32,3,16,covks=3)(res2_mp)
    res3_mp = tf.keras.layers.MaxPool1D(pool_size=2,name="res3_mp")(res3)
    res4 = ResBlock1D(64,4,32,covks=3)(res3_mp)
    res4_mp = tf.keras.layers.MaxPool1D(pool_size=2,name="res4_mp")(res4)
    res_enc = ResBlock1D(128,5,64,covks=3)(res4_mp)

    # ============================================ FFT Encoder =======================================================
    res21 = ResBlock1D(8,21,6)(FFT_layer)
    res21_mp = tf.keras.layers.MaxPool1D(pool_size=4,name="res21_mp")(res21)
    res22 = ResBlock1D(16,22,8,covks=3)(res21_mp)
    res22_mp = tf.keras.layers.MaxPool1D(pool_size=4,name="res22_mp")(res22)
    res23 = ResBlock1D(32,23,16,covks=3)(res22_mp)
    res23_mp = tf.keras.layers.MaxPool1D(pool_size=2,name="res23_mp")(res23)
    res24 = ResBlock1D(64,24,32,covks=3)(res23_mp)
    res24_mp = tf.keras.layers.MaxPool1D(pool_size=2,name="res24_mp")(res24)
    res_encfft = ResBlock1D(128,25,64,covks=3)(res24_mp)

    # ============================================= Dense ==============================================================
    ctfa_scale1 = CrossTFAttention(attnum=1, num_head=4, key_dim=16, output_shape=8)(frq=res21,sig=res1)
    ctfa_scale2 = CrossTFAttention(attnum=2, num_head=4, key_dim=16, output_shape=16)(frq=res22,sig=res2)
    ctfa_scale3 = CrossTFAttention(attnum=3, num_head=4, key_dim=16, output_shape=32)(frq=res23,sig=res3)
    ctfa_scale4 = CrossTFAttention(attnum=4, num_head=4, key_dim=16, output_shape=64)(frq=res24,sig=res4)
    # ctfa_scale_enc = CrossTFAttention(attnum=4, num_head=4, key_dim=16, output_shape=64)(frq=res_encfft,sig=res_enc)
    ctfa_scale_enc = CrossTFAttention(attnum=5, num_head=4, key_dim=16, output_shape=128)(frq=res_encfft,sig=res_enc)
    
    up5 = tf.keras.layers.UpSampling1D(size=64,name="up5")(ctfa_scale_enc)
    up4 = tf.keras.layers.UpSampling1D(size=32,name="up4")(ctfa_scale4)
    # up4 = tf.keras.layers.UpSampling1D(size=32,name="up4")(ctfa_scale_enc)
    up3 = tf.keras.layers.UpSampling1D(size=16,name="up3")(ctfa_scale3)
    up2 = tf.keras.layers.UpSampling1D(size=4,name="up2")(ctfa_scale2)

    up15 = tf.keras.layers.UpSampling1D(size=64,name="up15")(res_encfft)
    up14 = tf.keras.layers.UpSampling1D(size=32,name="up14")(res24)
    # up14 = tf.keras.layers.UpSampling1D(size=32,name="up14")(res_encfft)
    up13 = tf.keras.layers.UpSampling1D(size=16,name="up13")(res23)
    up12 = tf.keras.layers.UpSampling1D(size=4,name="up12")(res22)

    tenc_cc = tf.keras.layers.concatenate([ctfa_scale1,up2,up3,up4,up5],name="tenc_cc")
    fenc_cc = tf.keras.layers.concatenate([res21,up12,up13,up14,up15],name="fenc_cc")

    sig_deco = tf.keras.layers.MultiHeadAttention(num_heads=4,key_dim=16,kernel_initializer="he_normal",dropout=0.1,name="sig_deco")(tenc_cc,tenc_cc)
    sig_dec_res = tf.keras.layers.Add(name="sig_dec_res")([tenc_cc,sig_deco])
    sig_dec_ln = tf.keras.layers.LayerNormalization(name='sig_dec_ln')(sig_dec_res)
    sig_dec_ffn = tf.keras.layers.Dense(units=248,activation='gelu',kernel_initializer='he_normal',name='sig_dec_ffn')(sig_dec_ln)
    sig_dec_ffnres = tf.keras.layers.Add(name="sig_dec_ffnres")([sig_dec_ffn,sig_dec_ln])
    sig_dec_ffnln = tf.keras.layers.LayerNormalization(name='sig_dec_ffnln')(sig_dec_ffnres)
    
    sig_deco_gap = tf.keras.layers.GlobalAvgPool1D(name='sig_deco_gap')(sig_dec_ffnln)
    sig_deco_gmp = tf.keras.layers.GlobalMaxPool1D(name='sig_deco_gmp')(sig_dec_ffnln)
    sig_deco_cc = tf.keras.layers.concatenate([sig_deco_gap,sig_deco_gmp],name="sig_deco_cc")
    sig_dec_f = tf.keras.layers.Dense(units=16,name="sig_dec_f")(sig_deco_cc)
    sig_dec_gelu = tf.keras.layers.Activation(activation='tanh',name="sig_dec_gelu")(sig_dec_f)

    frq_deco = tf.keras.layers.MultiHeadAttention(num_heads=4,key_dim=16,kernel_initializer="he_normal",dropout=0.1,name="frq_deco")(fenc_cc,fenc_cc)
    frq_dec_res = tf.keras.layers.Add(name="frq_dec_res")([fenc_cc,frq_deco])
    frq_dec_ln = tf.keras.layers.LayerNormalization(name='frq_dec_ln')(frq_dec_res)
    frq_dec_ffn = tf.keras.layers.Dense(units=248,activation='gelu',kernel_initializer='he_normal',name='frq_dec_ffn')(frq_dec_ln)
    frq_dec_ffnres = tf.keras.layers.Add(name="frq_dec_ffnres")([frq_dec_ffn,frq_dec_ln])
    frq_dec_ffnln = tf.keras.layers.LayerNormalization(name='frq_dec_ffnln')(frq_dec_ffnres)
    
    frq_deco_gap = tf.keras.layers.GlobalAvgPool1D(name='frq_deco_gap')(frq_dec_ffnln)
    frq_deco_gmp = tf.keras.layers.GlobalMaxPool1D(name='frq_deco_gmp')(frq_dec_ffnln)
    frq_deco_cc = tf.keras.layers.concatenate([frq_deco_gap,frq_deco_gmp],name="frq_deco_cc")
    frq_dec_f = tf.keras.layers.Dense(units=16,name="frq_dec_f")(frq_deco_cc)
    frq_dec_gelu = tf.keras.layers.Activation(activation='tanh',name="frq_dec_gelu")(frq_dec_f)

    ft_cc = tf.keras.layers.concatenate([sig_deco_cc,frq_deco_cc],name="ft_cc")
    ft_fc = tf.keras.layers.Dense(units=16,activation='tanh',name="ft_fc")(ft_cc)
    # ft_add = tf.keras.layers.Add(name="ft_add")([ft_fc,sig_dec_gelu,frq_dec_gelu])

    sig_cls1 = tf.keras.layers.Dense(units=8,activation='softmax',name='sig_cls1')(sig_dec_gelu)
    sig_cls2 = tf.keras.layers.Dense(units=8,activation='softmax',name='sig_cls2')(sig_dec_gelu)
    sig_cls3 = tf.keras.layers.Dense(units=8,activation='softmax',name='sig_cls3')(sig_dec_gelu)

    frq_cls1 = tf.keras.layers.Dense(units=8,activation='softmax',name='frq_cls1')(frq_dec_gelu)
    frq_cls2 = tf.keras.layers.Dense(units=8,activation='softmax',name='frq_cls2')(frq_dec_gelu)
    frq_cls3 = tf.keras.layers.Dense(units=8,activation='softmax',name='frq_cls3')(frq_dec_gelu)

    mrg_cls1 = tf.keras.layers.Dense(units=8,activation='softmax',name='mrg_cls1')(ft_fc)
    mrg_cls2 = tf.keras.layers.Dense(units=8,activation='softmax',name='mrg_cls2')(ft_fc)
    mrg_cls3 = tf.keras.layers.Dense(units=8,activation='softmax',name='mrg_cls3')(ft_fc)

    runet_basemd = tf.keras.Model(inputs=ipl,outputs=[
                                                        sig_cls1,sig_cls2,sig_cls3,
                                                         frq_cls1,frq_cls2,frq_cls3,
                                                         mrg_cls1,mrg_cls2,mrg_cls3
                                                    ],name='runet_basemd')
    
    return runet_basemd

In [13]:
def EmoclsTeacher(bm: tf.keras.Model, cls=3) -> tf.keras.Model:
    ft_sig = bm.get_layer("sig_dec_gelu").output
    sig_cls = tf.keras.layers.Dense(units=cls,activation='softmax',name='sig_cls')(ft_sig)
    
    ft_frq = bm.get_layer("frq_dec_gelu").output
    frq_cls = tf.keras.layers.Dense(units=cls,activation='softmax',name='frq_cls')(ft_frq)

    ft_merg = bm.get_layer("ft_fc").output
    merg_cls = tf.keras.layers.Dense(units=cls,activation='softmax',name='merg_cls')(ft_merg)

    ft_cc = bm.get_layer("ft_cc").output
    emo_router = tf.keras.layers.Dense(units=3,activation='sigmoid',name='emo_router')(ft_cc)
    emo_vot_sig = tf.keras.layers.Multiply(name='emo_vot_sig')([emo_router[:,0],sig_cls])
    emo_vot_frq = tf.keras.layers.Multiply(name='emo_vot_frq')([emo_router[:,1],frq_cls])
    emo_vot_mrg = tf.keras.layers.Multiply(name='emo_vot_mrg')([emo_router[:,2],merg_cls])
    emo_vot_add = tf.keras.layers.Add(name='emo_vot_add')([emo_vot_sig,emo_vot_frq,emo_vot_mrg])
    # emo_cls = tf.keras.layers.Dense(units=cls,activation='softmax',name='emo_cls')(ft_add)
    emo_cls = tf.keras.layers.Activation(activation='softmax',name='emo_cls')(emo_vot_add)

    emocls = tf.keras.Model(inputs=bm.input,outputs=[emo_cls,ft_sig,ft_frq,ft_merg])
        
    return emocls

In [14]:
class MtsclEmb(tf.keras.layers.Layer):
    def __init__(self, resnum, **kwargs):
        super(MtsclEmb, self).__init__(name=f'MtsclEmb_{resnum}', **kwargs)

        self.scl1_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=1,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl1_emb{resnum}'
        )
    
        self.scl2_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=2,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl2_emb{resnum}'
        )

        self.scl3_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=3,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl3_emb{resnum}'
        )
      
        self.scl4_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=4,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl4_emb{resnum}'
        )

        self.scl5_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=5,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl5_emb{resnum}'
        )
    
        self.scl6_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=6,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl6_emb{resnum}'
        )

        self.scl7_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=7,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl7_emb{resnum}'
        )
      
        self.scl8_cov = tf.keras.layers.Conv1D(
            filters=2,
            kernel_size=3,
            padding='same',
            dilation_rate=8,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            name=f'scl8_emb{resnum}'
        )

        self.mscc = tf.keras.layers.Concatenate(name=f'mscc_emb{resnum}')
    
    def call(self, x):
        scl1 = self.scl1_cov(x)
        scl2 = self.scl2_cov(x)
        scl3 = self.scl3_cov(x)
        scl4 = self.scl4_cov(x)
        scl5 = self.scl5_cov(x)
        scl6 = self.scl6_cov(x)
        scl7 = self.scl7_cov(x)
        scl8 = self.scl8_cov(x)

        mscc = self.mscc([scl1,scl2,scl3,scl4,scl5,scl6,scl7,scl8])

        return mscc
        

In [15]:
class TransformerBlk(tf.keras.layers.Layer):
    def __init__(self, num_heads, key_dim, ffn_dim, **kwargs):
        super(TransformerBlk, self).__init__(name=f'TransformerBlk', **kwargs)
        self.mhatt = tf.keras.layers.MultiHeadAttention(num_heads=num_heads,key_dim=key_dim,dropout=0.1,kernel_initializer="he_normal",name="mhatt")
        self.add_att = tf.keras.layers.Add(name="add_att")
        self.ln_att = tf.keras.layers.LayerNormalization(name='ln_att')
        self.ffn = tf.keras.layers.Dense(units=ffn_dim,activation='gelu',kernel_initializer='he_normal',name='ffn_1')
        self.add_ffn = tf.keras.layers.Add(name="add_ffn")
        self.ln_ffn = tf.keras.layers.LayerNormalization(name='ln_ffn')

    def call(self, x) :
        mhatt = self.mhatt(x,x)
        add_att = self.add_att([x,mhatt])
        ln_att = self.ln_att(add_att)
        ffn = self.ffn(ln_att)
        add_ffn = self.add_ffn([ln_att,ffn])
        ln_ffn = self.ln_ffn(add_ffn)

        return ln_ffn


In [16]:
def EmoStumd(bs=32, fs: int=64, emo_num=3, distill=False) -> tf.keras.Model:
    # =========================================== signal layers resUnet ===============================================
    # =========================================== FFT layers resUnet ==================================================
    ipl = tf.keras.Input((20*fs,1),batch_size=bs)
    FFT_layer = FFTLayer1d()(ipl)
    # =========================================== signal Encoder ======================================================
    scl_sig = MtsclEmb(1)(ipl)

    # ============================================ FFT Encoder =======================================================
    scl_frq = MtsclEmb(2)(FFT_layer)

    tfb = TransformerBlk(num_heads=4,key_dim=16,ffn_dim=16)
    
    sig_tfb = tfb(scl_sig)
    frq_tfb = tfb(scl_frq)

    sig_deco_gap = tf.keras.layers.GlobalAvgPool1D(name='sig_deco_gap')(sig_tfb)
    sig_deco_gmp = tf.keras.layers.GlobalMaxPool1D(name='sig_deco_gmp')(sig_tfb)
    sig_deco_cc = tf.keras.layers.concatenate([sig_deco_gap,sig_deco_gmp],name="sig_deco_cc")
    sig_dec_f = tf.keras.layers.Dense(units=16,name="sig_dec_f")(sig_deco_cc)
    sig_dec_gelu = tf.keras.layers.Activation(activation='tanh',name="sig_dec_gelu")(sig_dec_f)

    frq_deco_gap = tf.keras.layers.GlobalAvgPool1D(name='frq_deco_gap')(frq_tfb)
    frq_deco_gmp = tf.keras.layers.GlobalMaxPool1D(name='frq_deco_gmp')(frq_tfb)
    frq_deco_cc = tf.keras.layers.concatenate([frq_deco_gap,frq_deco_gmp],name="frq_deco_cc")
    frq_dec_f = tf.keras.layers.Dense(units=16,name="frq_dec_f")(frq_deco_cc)
    frq_dec_gelu = tf.keras.layers.Activation(activation='tanh',name="frq_dec_gelu")(frq_dec_f)
    
    ft_cc = tf.keras.layers.concatenate([sig_deco_cc,frq_deco_cc],name="ft_cc")
    ft_fc = tf.keras.layers.Dense(units=16,activation='tanh',name="ft_fc")(ft_cc)

    emo_add = tf.keras.layers.Add(name='emo_add')([sig_dec_gelu,frq_dec_gelu,ft_fc])
    emo_cls = tf.keras.layers.Dense(units=emo_num,activation='softmax',name='emo_cls')(emo_add)

    if distill :
        runet_emostu = tf.keras.Model(inputs=ipl,outputs=[emo_cls,sig_dec_gelu,frq_dec_gelu,ft_fc],name='runet_emostu')
    else:
        runet_emostu = tf.keras.Model(inputs=ipl,outputs=emo_cls,name='runet_emostu')
        
    return runet_emostu

In [17]:
tst_emocls = EmoStumd()
print(tst_emocls.summary())

2025-07-15 14:25:24.145484: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38104 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB, pci bus id: 0000:19:00.0, compute capability: 8.0


Model: "runet_emostu"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (32, 1280, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fft_layer1d         │ (32, 1280, 2)     │          0 │ input_layer[0][0] │
│ (FFTLayer1d)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ MtsclEmb_1          │ (32, 1280, 16)    │         64 │ input_layer[0][0] │
│ (MtsclEmb)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ MtsclEmb_2          │ (32, 1280, 16)    │        112 │ fft_layer1d[0][0] │
│ (MtsclEmb)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ TransformerBlk      │ (32, 1280, 16)    │      4,640 │ MtsclEmb_1[0][0], │
│ (TransformerBlk)    │                   │            │ MtsclEmb_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sig_deco_gap        │ (32, 16)          │          0 │ TransformerBlk[0… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sig_deco_gmp        │ (32, 16)          │          0 │ TransformerBlk[0… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ frq_deco_gap        │ (32, 16)          │          0 │ TransformerBlk[1… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ frq_deco_gmp        │ (32, 16)          │          0 │ TransformerBlk[1… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sig_deco_cc         │ (32, 32)          │          0 │ sig_deco_gap[0][… │
│ (Concatenate)       │                   │            │ sig_deco_gmp[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ frq_deco_cc         │ (32, 32)          │          0 │ frq_deco_gap[0][… │
│ (Concatenate)       │                   │            │ frq_deco_gmp[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sig_dec_f (Dense)   │ (32, 16)          │        528 │ sig_deco_cc[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ frq_dec_f (Dense)   │ (32, 16)          │        528 │ frq_deco_cc[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ft_cc (Concatenate) │ (32, 64)          │          0 │ sig_deco_cc[0][0… │
│                     │                   │            │ frq_deco_cc[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sig_dec_gelu        │ (32, 16)          │          0 │ sig_dec_f[0][0]   │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ frq_dec_gelu        │ (32, 16)          │          0 │ frq_dec_f[0][0]   │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ft_fc (Dense)       │ (32, 16)          │      1,040 │ ft_cc[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emo_add (Add)       │ (32, 16)          │          0 │ sig_dec_gelu[0][

 Total params: 6,963 (27.20 KB)

 Trainable params: 6,963 (27.20 KB)

 Non-trainable params: 0 (0.00 B)

None


In [18]:
class SaveBestModel(tf.keras.callbacks.ModelCheckpoint):
    def __init__(self, filepath, verbose=0):
        super(SaveBestModel, self).__init__(filepath, verbose=0)
        self.best_acc_models = []
        self.best_f1 =0.0
        self.verbose = verbose

    def on_epoch_end(self, epoch, logs=None):
        val_acc = logs.get('val_categorical_accuracy')
        val_mf1 = logs.get('val_f1_score')
        
        if len(self.best_acc_models) < 3:
            self.best_acc_models.append(val_acc)
            if val_acc >= max(self.best_acc_models):
                self.best_f1 = val_mf1
                self.model.save(self.filepath)
                if self.verbose :
                    print(f"Save best model with val_categorical_accuracy={val_acc:.4f}, val_macro_f1_score={val_mf1:.4f}")
        else:
            self.best_acc_models.sort()
            if val_acc > self.best_acc_models[0]:
                # 检查当前的val_f1_score是否为最高
                if val_mf1 >= self.best_f1 and val_acc > (self.best_acc_models[2] - 0.01):
                    self.best_f1 = val_mf1
                    self.model.save(self.filepath)
                    if self.verbose :
                        print(f"Save best model with val_categorical_accuracy={val_acc:.4f}, val_macro_f1_score={val_mf1:.4f}")
                elif val_acc > self.best_acc_models[2] and val_mf1 >= (self.best_f1 - 0.025):
                    self.best_f1 = val_mf1
                    self.model.save(self.filepath)
                    if self.verbose :
                        print(f"Save best model with val_categorical_accuracy={val_acc:.4f}, val_macro_f1_score={val_mf1:.4f}")
                self.best_acc_models.pop(0)
                self.best_acc_models.append(val_acc)

In [19]:
@tf.function
def distillation_loss(y_true, y_pred, teacher_soft, sample_weight=None, temperature=3, alpha=0.15):
    
    # 硬标签损失（交叉熵）
    hard_loss = tf.keras.losses.CategoricalCrossentropy(from_logits=False)(y_true, y_pred, sample_weight=sample_weight)

    # 软标签损失（带温度的KL散度）
    # 应用温度后计算softmax
    student_soft = tf.nn.softmax(y_pred / temperature)
    teacher_soft = tf.nn.softmax(teacher_soft / temperature)
    
    # 计算KL散度
    soft_loss = tf.keras.losses.KLDivergence()(teacher_soft, student_soft, sample_weight=sample_weight)
    
    # 总损失 = α * 软损失 + (1 - α) * 硬损失
    return alpha * soft_loss + (1 - alpha) * hard_loss

In [20]:
def get_current_alpha(current_epoch,total_epochs,alpha_initial=0.6,alpha_final=0.15):
    # 计算当前alpha值，从alpha_initial线性衰减到alpha_final
    ratio = min(current_epoch / total_epochs, 1.0)
    current_alpha = tf.cast(alpha_initial - ratio * (alpha_initial - alpha_final),tf.float32)
    return current_alpha

In [21]:
class WarmUpCosineDecayRestarts(tf.keras.optimizers.schedules.LearningRateSchedule):
    """学习率预热加上余弦退火重启"""
    def __init__(
        self,
        initial_learning_rate,
        first_decay_steps,
        t_mul=2.0,
        m_mul=1.0,
        alpha=0.0,
        warm_step=1000,
        min_lr=0.0,
        max_lr=1e-3,
        name=None,
    ):
        super().__init__()
        self.initial_learning_rate = initial_learning_rate
        self.first_decay_steps = first_decay_steps
        self.t_mul = t_mul
        self.m_mul = m_mul
        self.alpha = alpha
        self.cosine_decay_restarts = tf.keras.optimizers.schedules.CosineDecayRestarts(
            initial_learning_rate=initial_learning_rate,
            first_decay_steps=first_decay_steps,
            t_mul=t_mul,
            m_mul=m_mul,
            alpha=alpha
        )
        self.warm_step = warm_step
        self.min_lr = min_lr
        self.max_lr = max_lr
        self.name = name

    def __call__(self, step):
        with tf.name_scope(self.name or "WarmUpCosineDecayRestarts"):
            # 使用 tf.cond 处理条件逻辑
            def warm_up_learning_rate():
                return tf.cast(
                    self.min_lr + (self.max_lr - self.min_lr) * step / self.warm_step,
                    tf.float32,
                )

            def cosine_decay_learning_rate():
                return self.cosine_decay_restarts(step - self.warm_step)

            return tf.cond(
                step < self.warm_step,
                warm_up_learning_rate,
                cosine_decay_learning_rate
            )

    def get_config(self):
        return {
            "initial_learning_rate": self.initial_learning_rate,
            "first_decay_steps": self.first_decay_steps,
            "t_mul": self.t_mul,
            "m_mul": self.m_mul,
            "alpha": self.alpha,
            "warm_step": self.warm_step,
            "min_lr": self.min_lr,
            "max_lr": self.max_lr,
            "name": self.name,
        }


# No distillation

In [ ]:
tf.keras.utils.set_random_seed(424)
epc = 50
bs = 32
lr = 1e-3

# ============================================================================= LOMO-CV =====================================================================================
for sub in np.arange(start=0,stop=30,step=1) :
    # print(f'Testing for Subject {sub+1} ...',flush=True)
    
    # ========================== label check ===================================
    train_lab = lab_seg[ann_seg[:,-1] != sub,4]
    test_lab = lab_seg[ann_seg[:,-1] == sub,4]
    
    train_onehot_lab = tf.keras.utils.to_categorical(train_lab, num_classes=3)
    test_onehot_lab = tf.keras.utils.to_categorical(test_lab, num_classes=3)
    spw = class_weight.compute_sample_weight(class_weight='balanced', y=train_lab)
    
    # =========================== data check ====================================
    train_sig = sig_mat[ann_seg[:,-1] != sub,::,0]
    test_sig = sig_mat[ann_seg[:,-1] == sub,::,0]
    
    # =========================== model check ====================================
    ptp_emocls = EmoStumd(fs=64,emo_num=3)
    
    opt_ptp = tf.keras.optimizers.AdamW(learning_rate=lr, clipnorm=1.)
    ptp_emocls.compile(loss={'emo_cls':'categorical_crossentropy'},
                       optimizer=opt_ptp,
                       metrics={'emo_cls':['categorical_accuracy',tf.keras.metrics.F1Score(average='macro'),tf.keras.metrics.F1Score(average='weighted')]})
    ckpt_ptp_fp = f"mout_distillation_res/case/avg_v3_a3/case_ptp_cls_loso_train_v3_s{sub}.valbest.keras"
    # ckpt_ptp = tf.keras.callbacks.ModelCheckpoint(ckpt_ptp_fp, monitor='val_categorical_accuracy', verbose=0, save_best_only=True, mode='auto')
    ckpt_ptp = SaveBestModel(filepath=ckpt_ptp_fp, verbose=0)
    _ = ptp_emocls.fit(x=train_sig[:,:,np.newaxis], y=train_onehot_lab, 
                             batch_size=bs, epochs=epc, verbose=0, sample_weight=spw,
                             validation_data=(test_sig[:,:,np.newaxis],test_onehot_lab),
                             callbacks=[ckpt_ptp])
    ptp_emocls.load_weights(f"mout_distillation_res/case/avg_v3_a3/case_ptp_cls_loso_train_v3_s{sub}.valbest.keras")
    # ptp_emocls.compile(loss={'emo_cls':'categorical_crossentropy'},
    #                    optimizer=opt_ptp,
    #                    metrics={'emo_cls':['categorical_accuracy',tf.keras.metrics.F1Score(average='macro'),tf.keras.metrics.F1Score(average='weighted')]})#
    loss,emo_acc,emo_f1,_,emo_f1w = ptp_emocls.evaluate(x=test_sig[:,:,np.newaxis],y=test_onehot_lab,verbose=0)
    print(f"Testing for Subject {sub+1} ... ACC={round(emo_acc,4)}, F1={round(emo_f1,4)}, F1W={round(emo_f1w,4)}")

# With distillation

In [ ]:
tf.keras.utils.set_random_seed(424)
epc = 75
bs = 32
lr = 1e-3

import time

lr_fn = WarmUpCosineDecayRestarts(
    initial_learning_rate = lr,
    first_decay_steps = 103*3,
    alpha=0.001,
    warm_step=206*10,
    min_lr=1e-8,
    max_lr=1e-3    
)

ptp_bsmdprtr = BMAutoEncoder()
teach_emocls = EmoclsTeacher(bm=ptp_bsmdprtr, cls=3)
opt_tch = tf.keras.optimizers.AdamW(learning_rate=1e-4, clipnorm=1.)
teach_emocls.compile(loss={'emo_cls':'categorical_crossentropy',
                           'sig_dec_gelu':'mean_squared_error','frq_dec_gelu':'mean_squared_error','ft_fc':'mean_squared_error'},
                     optimizer=opt_tch,
                     metrics={'emo_cls':["categorical_accuracy",tf.keras.metrics.F1Score(average="weighted")],
                              'sig_dec_gelu':'mean_absolute_error','frq_dec_gelu':'mean_absolute_error','ft_fc':'mean_absolute_error'})

for sub in np.arange(30) :
    # print(f'Testing for Subject {sub+1} ...',flush=True)
    start = time.time()
    # ========================== label check ===================================
    train_lab = lab_seg[ann_seg[:,-1] != sub,4]
    test_lab = lab_seg[ann_seg[:,-1] == sub,4]
    
    train_onehot_lab = tf.keras.utils.to_categorical(train_lab, num_classes=3)
    test_onehot_lab = tf.keras.utils.to_categorical(test_lab, num_classes=3)
    spw = class_weight.compute_sample_weight(class_weight='balanced', y=train_lab)
    
    # =========================== data check ====================================
    train_sig = sig_mat[ann_seg[:,-1] != sub,::,0][:,:,np.newaxis]
    test_sig = sig_mat[ann_seg[:,-1] == sub,::,0][:,:,np.newaxis]

    train_sig_3md = sig_mat[ann_seg[:,-1] != sub,::,:]
    test_sig_3md = sig_mat[ann_seg[:,-1] == sub,::,:]
    
    # =========================== teacher model check ====================================
    teach_emocls.load_weights(f"case_md_mout_res/avg_v3_a3/case_ptp_cls_loso_train_v3_s{sub}.valbest.keras")
    
    stu_emocls = EmoStumd(bs=bs,fs=64,emo_num=3,distill=True)
    stu_emocls.load_weights(f"mout_distillation_res/case/avg_v3_a3/case_ptp_cls_loso_train_v3_s{sub}.valbest.keras")
    opt_stu = tf.keras.optimizers.AdamW(learning_rate=lr, clipnorm=1.)
    ckpt_ptp_fp = f"mout_distillation_res/case_distill/avg_v3_a3_ftsh/case_ptp_cls_loso_train_v3_s{sub}.valbest.keras"
    # ckpt_ptp = SaveBestF1Model(filepath=ckpt_ptp_fp,verbose=2)
    # ckpt_ptp = tf.keras.callbacks.ModelCheckpoint(ckpt_ptp_fp, monitor='val_categorical_accuracy', verbose=0, save_best_only=True, mode='auto')

    cea = tf.keras.metrics.CategoricalAccuracy()
    mf1 = tf.keras.metrics.F1Score(average="macro")
    wf1 = tf.keras.metrics.F1Score(average="weighted")
    
    stu_md_trainable_variables = stu_emocls.trainable_variables
   
    @tf.function
    def train_step(stu_md, tch_md, stu_x, stu_y, tch_x, opt_stu, cea, mf1, wf1, sample_weight, temperature=3, alpha=0.15):
        tch_md.trainable = False
        teacher_emo_cls, teacher_sig, teacher_frq, teacher_fc = tch_md(tch_x)
        with tf.GradientTape() as tape:
            stu_md.trainable = True
            student_emo_cls, sig_dec_gelu,frq_dec_gelu,ft_fc = stu_md(stu_x)
            distill_loss_value = distillation_loss(y_true=stu_y, y_pred=student_emo_cls, teacher_soft=teacher_emo_cls, 
                                                   sample_weight=sample_weight, temperature=temperature, alpha=alpha)
            # ftmse_loss_value = 1.5*alpha*(
            #     0.5*tf.keras.losses.KLDivergence()(y_true=teacher_ft_fc,y_pred=student_ft_fc,sample_weight=sample_weight) + 
            #     (1-tf.abs(tf.keras.losses.CosineSimilarity()(y_true=teacher_ft_fc,y_pred=student_ft_fc,sample_weight=sample_weight)))
            # )
    
            sig_ft_loss = 1.5*alpha*tf.keras.losses.MeanSquaredError()(y_true=teacher_sig,y_pred=sig_dec_gelu,sample_weight=sample_weight)
            frq_ft_loss = 1.5*alpha*tf.keras.losses.MeanSquaredError()(y_true=teacher_frq,y_pred=frq_dec_gelu,sample_weight=sample_weight)
            mrg_ft_loss = 1.5*alpha*tf.keras.losses.MeanSquaredError()(y_true=teacher_fc,y_pred=ft_fc,sample_weight=sample_weight)
    
            gradients = tape.gradient([distill_loss_value, sig_ft_loss, frq_ft_loss, mrg_ft_loss], stu_md_trainable_variables)
            opt_stu.apply_gradients(zip(gradients, stu_md_trainable_variables))
    
        cea.update_state(stu_y,student_emo_cls)
        wf1.update_state(stu_y,student_emo_cls)
        mf1.update_state(stu_y,student_emo_cls)
        
        # 返回指标结果
        train_dic = {'emo_cls_distill_loss':distill_loss_value,
                     'emo_cls_categorical_accuracy':cea.result(),
                     'emo_cls_macro_f1_score':mf1.result(),
                     'emo_cls_weighted_f1_score':wf1.result(),
                     'sig_dec_gelu_loss':sig_ft_loss,
                    'frq_dec_gelu_loss':frq_ft_loss,
                    'ft_cc_loss':mrg_ft_loss}
        return train_dic
    
    
    @tf.function
    def test_step(stu_md, tch_md, stu_vx, stu_vy, tch_vx, cea, mf1, wf1, temperature=3, alpha=0.15):
        tch_md.trainable = False
        stu_md.trainable = False
        teacher_emo_cls_v, teacher_sig, teacher_frq, teacher_fc = tch_md(tch_vx)
        student_emo_cls_v, sig_dec_gelu, frq_dec_gelu, ft_fc = stu_md(stu_vx)
    
        distill_loss_value = distillation_loss(y_true=stu_vy, y_pred=student_emo_cls_v, teacher_soft=teacher_emo_cls_v, temperature=temperature, alpha=alpha)
        # ftmse_loss_value = 1.5*alpha*(
        #     0.5*tf.keras.losses.KLDivergence()(y_true=teacher_ft_fc_v,y_pred=student_ft_fc_v) + 
        #     (1-tf.abs(tf.keras.losses.CosineSimilarity()(y_true=teacher_ft_fc_v,y_pred=student_ft_fc_v)))
        # )
    
        sig_ft_loss = 1.5*alpha*tf.keras.losses.MeanSquaredError()(y_true=teacher_sig,y_pred=sig_dec_gelu)
        frq_ft_loss = 1.5*alpha*tf.keras.losses.MeanSquaredError()(y_true=teacher_frq,y_pred=frq_dec_gelu)
        mrg_ft_loss = 1.5*alpha*tf.keras.losses.MeanSquaredError()(y_true=teacher_fc,y_pred=ft_fc)
    
        cea.update_state(stu_vy,student_emo_cls_v)
        wf1.update_state(stu_vy,student_emo_cls_v)
        mf1.update_state(stu_vy,student_emo_cls_v)
        
        # 返回指标结果
        test_dic = {'val_emo_cls_distill_loss':distill_loss_value,
                     'val_emo_cls_categorical_accuracy':cea.result(),
                     'val_emo_cls_macro_f1_score':mf1.result(),
                     'val_emo_cls_weighted_f1_score':wf1.result(),
                    'val_sig_dec_gelu_loss':sig_ft_loss,
                    'val_frq_dec_gelu_loss':frq_ft_loss,
                    'val_ft_cc_loss':mrg_ft_loss}
        return test_dic 
    
    n_bs = int(len(train_sig[:,0,0])/bs)+1
    best_f1 = 0
    best_acc_models = []       
    for epoch in np.arange(epc) :
        # print(f"\n Epoch {epoch+1}/{epc}:")
        for part in np.arange(n_bs) :
            chose_idxn = np.random.choice(len(train_sig[:,0,0]),bs,replace=False)
            alpha_used = get_current_alpha(current_epoch=epoch,total_epochs=epc)
            train_dic = train_step(stu_md=stu_emocls,tch_md=teach_emocls,
                                   stu_x=train_sig[chose_idxn,:,:],stu_y=train_onehot_lab[chose_idxn,:],tch_x=train_sig_3md[chose_idxn,:,:],
                                   opt_stu=opt_stu, cea=cea, mf1=mf1, wf1=wf1,
                                   sample_weight=spw[chose_idxn], temperature=3,alpha=alpha_used)
        
        cea.reset_state()
        mf1.reset_state()
        wf1.reset_state()
        
        test_dic = test_step(stu_md=stu_emocls, tch_md=teach_emocls,
                             stu_vx=test_sig, stu_vy=test_onehot_lab, tch_vx=test_sig_3md,
                             cea=cea, mf1=mf1, wf1=wf1,
                             temperature=3, alpha=alpha_used)
        
        # print(f"emo_cls_distill_loss={(train_dic['emo_cls_distill_loss']).numpy():.4f}, emo_cls_categorical_accuracy={(train_dic['emo_cls_categorical_accuracy']).numpy():.4f}, emo_cls_macro_f1_score={(train_dic['emo_cls_macro_f1_score']).numpy():.4f}, emo_cls_weighted_f1_score={(train_dic['emo_cls_weighted_f1_score']).numpy():.4f}, "+
        #       f"val_emo_cls_distill_loss={(test_dic['val_emo_cls_distill_loss']).numpy():.4f}, val_emo_cls_categorical_accuracy={test_dic['val_emo_cls_categorical_accuracy'].numpy():.4f}, val_emo_cls_macro_f1_score={test_dic['val_emo_cls_macro_f1_score'].numpy():.4f}, val_emo_cls_weighted_f1_score={test_dic['val_emo_cls_weighted_f1_score'].numpy():.4f}")

        val_mf1 = test_dic['val_emo_cls_macro_f1_score'].numpy()
        val_acc = test_dic['val_emo_cls_categorical_accuracy'].numpy()
        
        if len(best_acc_models) < 3:
            best_acc_models.append(val_acc)
            if val_acc >= max(best_acc_models[:]) :
                best_f1 = val_mf1
                # 保存当前最佳模型为.keras格式
                stu_emocls.save(ckpt_ptp_fp)
                # print(f"Save best model with val_categorical_accuracy={test_dic['val_emo_cls_categorical_accuracy'].numpy():.4f}, val_macro_f1_score={test_dic['val_emo_cls_macro_f1_score'].numpy():.4f}, val_weighted_f1_score={test_dic['val_emo_cls_weighted_f1_score'].numpy():.4f}")
        else:
            best_acc_models.sort()
            if val_acc > best_acc_models[0]:
                # 检查当前的val_f1_score是否为最高
                if (val_mf1 >= best_f1) and (val_acc > (best_acc_models[2] - 0.01)):
                    best_f1 = val_mf1
                    # 保存当前最佳模型为.keras格式
                    stu_emocls.save(ckpt_ptp_fp)
                    # print(f"Save best model with val_categorical_accuracy={test_dic['val_emo_cls_categorical_accuracy'].numpy():.4f}, val_macro_f1_score={test_dic['val_emo_cls_macro_f1_score'].numpy():.4f}, val_weighted_f1_score={test_dic['val_emo_cls_weighted_f1_score'].numpy():.4f}")
                elif (val_acc > best_acc_models[2]) and (val_mf1 >= (best_f1 - 0.025)):
                    best_f1 = val_mf1
                    stu_emocls.save(ckpt_ptp_fp)
                    # print(f"Save best model with val_categorical_accuracy={test_dic['val_emo_cls_categorical_accuracy'].numpy():.4f}, val_macro_f1_score={test_dic['val_emo_cls_macro_f1_score'].numpy():.4f}, val_weighted_f1_score={test_dic['val_emo_cls_weighted_f1_score'].numpy():.4f}")
                best_acc_models.pop(0)
                best_acc_models.append(val_acc)
                
        cea.reset_state()
        mf1.reset_state()
        wf1.reset_state()
        
    tst_sm_emocls = EmoStumd(bs=bs,fs=64,emo_num=3,distill=False)
    tst_sm_emocls.load_weights(ckpt_ptp_fp)
    tst_sm_emocls.compile(loss={'emo_cls':'categorical_crossentropy'},
                         optimizer=opt_stu,
                         metrics={'emo_cls':["categorical_accuracy",tf.keras.metrics.F1Score(average="macro"),tf.keras.metrics.F1Score(average="weighted")]})    
    loss,acc,f1,_,f1w = tst_sm_emocls.evaluate(test_sig[:,:,np.newaxis], test_onehot_lab, verbose=0)
    end = time.time()
    print(f"Testing for Subject {sub+1} ... -{end - start:.4f} s-,  ACC={acc:.4f},F1={f1:.4f},W-F1={f1w:.4f}")
